Read files from cloud storage by datastream autoloader

In [0]:
%python
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    TimestampType
)

customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("member_since", DateType(), True),
    StructField("created_timestamp", TimestampType(), True)
])

customer_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/project_etl/landing/opertaional/customer_autoloader/_checkpoint_autoloader")
    .schema(customer_schema)
    .load("/Volumes/project_etl/landing/opertaional/customer_autoloader")
)


In [0]:
%python
from pyspark.sql.functions import current_timestamp

cust_df_add = (
    customer_df
    .withColumn("loading_ts", current_timestamp())
)


In [0]:
%python
cust_df_add.writeStream \
    .format("delta") \
    .option(
        "checkpointLocation",
        "/Volumes/project_etl/landing/opertaional/customer_autoloader/_checkpoint_stream"
    ) \
    .trigger(availableNow=True) \
    .toTable("project_etl.bronze.customer_autoloader")